# **ETL Pipeline — Car Price Dataset**

## Objectives

* Load the raw car price dataset from the inputs folder
* Clean the data by handling missing values and encoding categorical variables
* Engineer new features such as price-to-performance ratio
* Save the cleaned dataset to the outputs folder



## Inputs
* inputs/carprice_dataset/CarPrice_Assignment.csv


## Outputs
* outputs/datasets/cleaned/car_prices_cleaned.csv

---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [3]:
import os
current_dir = os.getcwd()
current_dir

'/Users/tildeholmqvist/Documents/VS_Code_Tilde/DA_project_2/jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [4]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [5]:
current_dir = os.getcwd()
current_dir

'/Users/tildeholmqvist/Documents/VS_Code_Tilde/DA_project_2'

# Section 1 — Load Data

In [6]:
import pandas as pd
import numpy as np

df = pd.read_csv('inputs/carprice_dataset/CarPrice_Assignment.csv')
df.head()

,car_ID,symboling,CarName,fueltype,aspiration,doornumber,carbody,drivewheel,enginelocation,wheelbase,...,enginesize,fuelsystem,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price
0,1,3,alfa-romero giulia,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,13495.0
1,2,3,alfa-romero stelvio,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,16500.0
2,3,1,alfa-romero Quadrifoglio,gas,std,two,hatchback,rwd,front,94.5,...,152,mpfi,2.68,3.47,9.0,154,5000,19,26,16500.0
3,4,2,audi 100 ls,gas,std,four,sedan,fwd,front,99.8,...,109,mpfi,3.19,3.40,10.0,102,5500,24,30,13950.0
4,5,2,audi 100ls,gas,std,four,sedan,4wd,front,99.4,...,136,mpfi,3.19,3.40,8.0,115,5500,18,22,17450.0


In [7]:
df.filter(['fueltype'], axis=1).value_counts()

fueltype
gas         185
diesel       20
Name: count, dtype: int64

---

In [8]:
print(df.shape)
print(df.dtypes)
df.isnull().sum()

(205, 26)
car_ID                int64
symboling             int64
CarName              object
fueltype             object
aspiration           object
doornumber           object
carbody              object
drivewheel           object
enginelocation       object
wheelbase           float64
carlength           float64
carwidth            float64
carheight           float64
curbweight            int64
enginetype           object
cylindernumber       object
enginesize            int64
fuelsystem           object
boreratio           float64
stroke              float64
compressionratio    float64
horsepower            int64
peakrpm               int64
citympg               int64
highwaympg            int64
price               float64
dtype: object


car_ID              0
symboling           0
CarName             0
fueltype            0
aspiration          0
doornumber          0
carbody             0
drivewheel          0
enginelocation      0
wheelbase           0
carlength           0
carwidth            0
carheight           0
curbweight          0
enginetype          0
cylindernumber      0
enginesize          0
fuelsystem          0
boreratio           0
stroke              0
compressionratio    0
horsepower          0
peakrpm             0
citympg             0
highwaympg          0
price               0
dtype: int64

## Data Overview
* 205 rows and 26 columns
* No missing values found
* Target variable: price (float64)
* Mix of numerical and categorical features

In [9]:
categorical_cols = df.select_dtypes(include='object').columns
for col in categorical_cols:
    print(f"{col}: {df[col].nunique()} unique values — {df[col].unique()}")

CarName: 147 unique values — ['alfa-romero giulia' 'alfa-romero stelvio' 'alfa-romero Quadrifoglio'
 'audi 100 ls' 'audi 100ls' 'audi fox' 'audi 5000' 'audi 4000'
 'audi 5000s (diesel)' 'bmw 320i' 'bmw x1' 'bmw x3' 'bmw z4' 'bmw x4'
 'bmw x5' 'chevrolet impala' 'chevrolet monte carlo' 'chevrolet vega 2300'
 'dodge rampage' 'dodge challenger se' 'dodge d200' 'dodge monaco (sw)'
 'dodge colt hardtop' 'dodge colt (sw)' 'dodge coronet custom'
 'dodge dart custom' 'dodge coronet custom (sw)' 'honda civic'
 'honda civic cvcc' 'honda accord cvcc' 'honda accord lx'
 'honda civic 1500 gl' 'honda accord' 'honda civic 1300' 'honda prelude'
 'honda civic (auto)' 'isuzu MU-X' 'isuzu D-Max ' 'isuzu D-Max V-Cross'
 'jaguar xj' 'jaguar xf' 'jaguar xk' 'maxda rx3' 'maxda glc deluxe'
 'mazda rx2 coupe' 'mazda rx-4' 'mazda glc deluxe' 'mazda 626' 'mazda glc'
 'mazda rx-7 gs' 'mazda glc 4' 'mazda glc custom l' 'mazda glc custom'
 'buick electra 225 custom' 'buick century luxus (sw)' 'buick century'
 'buic

## Categorical Features Analysis
* CarName has 147 unique values — needs cleaning (typos found: 'maxda' instead of 'mazda', 'porcshce' instead of 'porsche', 'toyouta' instead of 'toyota', 'vokswagen' instead of 'volkswagen')
* Most categorical features have few unique values and are suitable for encoding
* cylindernumber is stored as text — needs converting to numbers

# OBS - move to ML
## Note — CarBrand Extraction and Cleaning
The CarBrand extraction and typo corrections below will be moved to the 
ML pipeline in notebook 03_ML.ipynb. This ensures the transformations 
are applied consistently during model training and prediction, rather 
than being applied manually before the pipeline.

In [20]:
#df['CarBrand'] = df['CarName'].str.split().str[0].str.lower()
#print(df['CarBrand'].unique())

### PIPELINE - fixa 

In [21]:
# df['CarBrand'] = df['CarBrand'].replace({
#     'maxda': 'mazda',
#     'porcshce': 'porsche',
#     'toyouta': 'toyota',
#     'vokswagen': 'volkswagen',
#     'vw': 'volkswagen'
# })

# print(df['CarBrand'].unique())
# print(f"Unique brands: {df['CarBrand'].nunique()}")

In [12]:
print(df['price'].describe())
print(f"\nMean price: {df['price'].mean():.2f}")
print(f"Median price: {df['price'].median():.2f}")
print(f"Std deviation: {df['price'].std():.2f}")

count      205.000000
mean     13276.710571
std       7988.852332
min       5118.000000
25%       7788.000000
50%      10295.000000
75%      16503.000000
max      45400.000000
Name: price, dtype: float64

Mean price: 13276.71
Median price: 10295.00
Std deviation: 7988.85


## Price Statistics
* Mean price: $13,276 — but higher than median, suggesting right skew
* Median price: $10,295 — better measure of "typical" car price due to outliers
* Std deviation: $7,988 — high spread indicating large price variation
* Min: $5,118 / Max: $45,400 — wide price range
* The difference between mean and median suggests positive skewness (expensive cars pulling the average up)

In [13]:
cylinder_map = {
    'two': 2, 'three': 3, 'four': 4,
    'five': 5, 'six': 6, 'eight': 8, 'twelve': 12
}
df['cylindernumber'] = df['cylindernumber'].map(cylinder_map)

In [14]:
df['doornumber'] = df['doornumber'].map({'two': 2, 'four': 4})

In [15]:
df['price_per_horsepower'] = df['price'] / df['horsepower']

In [16]:
df['price_per_enginesize'] = df['price'] / df['enginesize']

## Feature Engineering

**price_per_horsepower** — selected as horsepower is a widely recognised performance metric in the automotive industry, providing a direct measure of price efficiency relative to performance.

**price_per_enginesize** — engine size is a key technical specification that influences manufacturing costs and performance, making it a relevant factor in understanding price variation.

In [17]:
Q1 = df['price'].quantile(0.25)
Q3 = df['price'].quantile(0.75)
IQR = Q3 - Q1
outliers = df[(df['price'] < Q1 - 1.5*IQR) | (df['price'] > Q3 + 1.5*IQR)]
print(f"Number of price outliers: {len(outliers)}")

Number of price outliers: 15


* 15 price outliers identified — retained in dataset as they represent legitimate luxury/high-end vehicles, not data errors

## Cleaning Summary
* Extracted CarBrand from CarName
* Note: CarBrand typo corrections moved to 03_ML.ipynb as per lecturer feedback
* Converted cylindernumber and doornumber from text to integers (ordinal encoding)
* Created new features: price_per_horsepower and price_per_enginesize
* 15 price outliers identified using IQR method — retained as legitimate luxury vehicles

---

In [18]:
import os
os.makedirs('outputs/datasets/cleaned', exist_ok=True)
df.to_csv('outputs/datasets/cleaned/car_prices_cleaned.csv', index=False)

---

# Push files to Repo

* In cases where you don't need to push files to Repo, you may replace this section with "Conclusions and Next Steps" and state your conclusions and next steps.

In [1]:
import os
try:
    os.makedirs(name='outputs/datasets/cleaned', exist_ok=True)
    print("Directory created or already exists")
except Exception as e:
    print(e)

Directory created or already exists


---

# Conclusions and Next Steps
* ETL pipeline complete — raw data successfully cleaned and saved
* Dataset ready for exploratory data analysis (EDA)
* Next: 02_EDA.ipynb